# Multi-date document training and backtest scoring

The active run uses `calendar_quarter_prefix_v1`: fixed calendar-quarter boundaries, start-history plus all native updates, stable positions with right padding, and causal per-date issuer fusion. Training and inference share this contract. All feature/subtoken/token NTP and MTP tasks remain; supervised labels stay on actual event dates. Inputs and targets retain their existing availability rules.

The $1T functional smoke trained three batches, then compared January-only predictions with January inside a whole-quarter pass. The $100B timing smoke trained three batches and scored **all 96,181 requested dates**, then completed the original six yearly long/short books. This is a speed and coverage experiment, not a claim about returns from a three-batch model. A separate fresh 12-epoch $100B run is active.

See [the contract and validation](../docs/multirate-document-scoring.md). The older July implementation inspired exporting all daily token scores per document. Its pandas input construction was not restored. Field layouts and task names are identical to the prior expanded model; the document context and trained weights differ.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys
repo = Path.cwd()
if repo.name == 'notebooks':
    repo = repo.parent
os.chdir(repo)
universe = '100B'
root = repo / 'artifacts/multirate_recovery' / universe
train_command = json.loads((root / 'training_command_documents_v10.json').read_text())
benchmark_command = json.loads((root / 'documents_full_calendar_command_v10.json').read_text())
assert '--checkpoint' not in train_command and '--resume-training' not in train_command
assert train_command[train_command.index('--sequence-mode') + 1] == 'documents'


In [ ]:
# Exact recorded smoke recipe: same corpus, all requested dates, FP32 CUDA.
# Separate processes avoid changing the current run. Choose a fresh output path.
RUN_BENCHMARK = False
if RUN_BENCHMARK:
    destination = root / 'documents_full_calendar_replay'
    if destination.exists():
        raise ValueError('Choose a new output directory; preserve earlier artifacts')
    command = list(benchmark_command)
    command[command.index('--output-dir') + 1] = str(destination)
    subprocess.run(command, check=True, env={**os.environ,
        'PYTHONPATH': str(repo) + os.pathsep + str(repo.parent / 'quant-warehouse')})
    from quant_orchestrator.research_tools.epoch_evaluation import yearly_epoch_backtests
    yearly_epoch_backtests(command, destination, '2024-01-02', '2026-09-09')

# The active fresh run was launched by the recorded supervisor below. It reads
# training_command_documents_v10.json, starts the trainer plus epoch monitor,
# preserves the original frozen adjusted prices, and blocks at each epoch gate.
# Inspect rather than relaunching into the existing output directory.
supervisor_source = '"""Fresh model training with a blocking epoch backtest monitor."""\nimport json,subprocess,sys,time,traceback\nfrom pathlib import Path\nroot=Path(sys.argv[1]).resolve();repo=Path.cwd();output=root/\'train_documents_v10\';output.mkdir(exist_ok=False)\nevaluation=output/\'epoch_validation_2024_2026\';evaluation.mkdir()\ncommand_file=root/\'training_command_documents_v10.json\'\nbase=json.loads(command_file.read_text())\nassert \'--checkpoint\' not in base and \'--resume-training\' not in base\nimport shutil, os\nshutil.copytree(root/\'train_expanded_v9/epoch_validation_2024_2026/backtest_prices\', evaluation/\'backtest_prices\', copy_function=os.link)\n(output/\'launch_manifest.json\').write_text(json.dumps(dict(initialization=\'fresh random model weights and fresh optimizer\',checkpoint=None,resume=False,epochs=12,min_market_cap=100000000000,training_cutoff=\'2024-01-01\',backtest_periods=[\'2024\',\'2025\',\'2026_YTD_through_2026-09-09\'],blocking_epoch_backtests=True,source_smoke_run=str(root.parent/"1T/train_documents_smoke_v10"),document_contract=\'calendar_quarter_prefix_v1\',full_calendar_benchmark=str(root/\'documents_full_calendar_timing_v10.json\'),event_target_date="transaction_date",event_input_date="disclosure_date",command=base),indent=2))\ndef status(stage,**kw):\n p=root/\'run_status.tmp\';p.write_text(json.dumps(dict(stage=stage,supervisor_pid=__import__(\'os\').getpid(),output=str(output),from_scratch=True,synchronous_epoch_backtests=True,**kw),indent=2));p.replace(root/\'run_status.json\')\ntrainer=monitor=None\ntry:\n with (root/\'training_documents_v10.log\').open(\'w\') as training_log,(root/\'epoch_validation_documents_v10.log\').open(\'w\') as evaluation_log:\n  trainer=subprocess.Popen(base+[\'--skip-predictions\'],stdout=training_log,stderr=subprocess.STDOUT)\n  monitor=subprocess.Popen([sys.executable,str(repo/\'scripts/monitor_multirate_epochs.py\'),\'--command-file\',str(command_file),\'--training-log\',str(root/\'training_documents_v10.log\'),\'--output-dir\',str(evaluation),\'--validation-start\',\'2024-01-02\',\'--validation-end\',\'2026-09-09\',\'--inference-corpus\',str(root/\'corpus_expanded_v9\'),\'--backtest-by-year\',\'--training-pid\',str(trainer.pid),\'--backtest-anchored-hits\'],stdout=evaluation_log,stderr=subprocess.STDOUT)\n  status(\'fresh_training_with_epoch_backtests\',pid=trainer.pid,monitor_pid=monitor.pid)\n  while trainer.poll() is None:\n   if monitor.poll() is not None:raise RuntimeError(\'Epoch monitor exited while training was active\')\n   time.sleep(5)\n  if trainer.returncode:raise RuntimeError(\'Training failed; inspect training_documents_v10.log\')\n  if monitor.wait():raise RuntimeError(\'Epoch monitor failed\')\n status(\'fresh_training_and_all_epoch_backtests_complete\')\nexcept Exception as exc:\n (evaluation/\'failure.json\').write_text(json.dumps({\'error\':str(exc)}))\n for child in (trainer,monitor):\n  if child is not None and child.poll() is None:child.terminate()\n status(\'failed\',error=str(exc));traceback.print_exc();raise\n'


In [1]:
import json
from pathlib import Path
root=Path('artifacts/multirate_recovery/100B')
old=json.loads((root/'train_expanded_v9/epoch_validation_2024_2026/epoch_0001/epoch_metrics.json').read_text())
old_model=json.loads((root/'train_expanded_v9/epoch_validation_2024_2026/epoch_0001/training_summary.json').read_text())
new_model=json.loads((root/'documents_full_calendar_benchmark_v10/training_summary.json').read_text())
timing=json.loads((root/'documents_full_calendar_timing_v10.json').read_text())
coverage=json.loads((root/'documents_full_calendar_benchmark_v10/prediction_coverage.json').read_text())
for key in ['feature_families','feature_family_dimensions','sparse_input_families','family_labels','tasks']:
    assert old_model[key]==new_model[key], key
assert coverage['prediction_rows']==96181
report=dict(old_inference_seconds=old['inference_seconds'],document_inference_seconds=timing['wall_seconds'],
    inference_speedup=old['inference_seconds']/timing['wall_seconds'],document_portfolio_seconds=timing['backtest_seconds'],
    document_total_seconds=timing['wall_seconds']+timing['backtest_seconds'],
    old_scoring_windows=old_model['evaluation_samples'],document_scoring_windows=new_model['evaluation_samples'],
    numeric_fields=len(new_model['feature_families']),numeric_groups=len(new_model['feature_family_dimensions']),
    sparse_input_families=len(new_model['sparse_input_families']),tasks=len(new_model['tasks']),layouts_identical=True,
    coverage=coverage,benchmark_weights='fresh model after three optimizer steps; timing/coverage only',
    prefix_validation=json.loads((root.parent/'1T/document_prefix_validation_v10.json').read_text()))
(root/'document_run_comparison_v10.json').write_text(json.dumps(report,indent=2))
print(json.dumps(report,indent=2))


{
  "old_inference_seconds": 2435.8582216659997,
  "document_inference_seconds": 66.35010839899769,
  "inference_speedup": 36.712196565195015,
  "document_portfolio_seconds": 2.8436729660024866,
  "document_total_seconds": 69.19378136500018,
  "old_scoring_windows": 96181,
  "document_scoring_windows": 1292,
  "numeric_fields": 1383,
  "numeric_groups": 45,
  "sparse_input_families": 15,
  "tasks": 42,
  "layouts_identical": true,
  "coverage": {
    "prediction_rows": 96181,
    "duplicate_keys": 0,
    "missing_dates": 0,
    "unexpected_dates": 0,
    "nonfinite_rows": 0
  },
  "benchmark_weights": "fresh model after three optimizer steps; timing/coverage only",
  "prefix_validation": {
    "prefix_rows": 371,
    "supervised_heads": 26,
    "maximum_absolute_difference": 8.344650268554688e-07,
    "prefix_coverage": {
      "prediction_rows": 371,
      "duplicate_keys": 0,
      "missing_dates": 0,
      "unexpected_dates": 0,
      "nonfinite_rows": 0
    },
    "quarter_coverage